# 数学目标与风险约束：基础审查

## tl;dr

- 建议取消未被问卷识别的 **λ=10/4/1**，以客户期限下的年化期望对数增长为目标，并显式约束损失超限概率。
- **α=10% 只作为公开的基准服务政策**。5%、20% 是半倍/双倍敏感性情形，既非估计出的偏好区间，也非统计置信区间。
- 120 条独立评价路径中 12 条超限，单侧 95% 精确概率上界约为 **15.70%**；即便是这个区间，也不适用于同样本挑选后未经修正的认证。
- 概率约束不限制超限严重程度；同时展示平均超额回撤和 CVaR，避免把频率与幅度混为一谈。

这是一份基础研究记录，不是竞赛提交报告。本轮不选择权重、产品映射或预测参数。

## Context & Methods

读者：队长与后续模型实现者。决策：选择数学目标，解释风险门槛，并识别现有信息的边界。

### Key Assumptions

1. 问卷 $H,L,\ell$ 分别为期限、主观可忍受的暂时购买力损失幅度、流动性等级；没有超限概率或收益—风险无差异点。
2. 策略须遵守因果决策、题定费用、初始票据持仓、无杠杆和全额投资条件。推荐时评价三款产品各自的实际执行规则。
3. 年度净实际财富为 $R_t=W_t/\prod_{s=1}^t(1+i_s)$，$R_0=100$。年度 MDD 遗漏年内下跌；票据也不保证购买力。
4. 下方反例均为构造数据，不是市场表现。二项示例仅适用于固定规则、独立同分布评价路径，且未在这些结果上选择规则。
5. V1 最终场景已被评估和查看，后续不能称拥有全新盲测。本次没有读取任何市场路径或旧实验结果。

小规模精确数值表与财富路径已完整呈现比较，故不另绘图。

## Data

本次唯一数值输入是数学常数、旧 λ 的公开取值与下方列明的构造路径。计算函数在 `scripts/audit_objective_risk.py`，不加载市场 CSV，也不导入策略引擎。

重新执行：安装项目依赖及 `requirements-review.txt`，为所用环境选择 Jupyter 内核；从仓库根目录或 `notebooks/` 运行。本笔记的中间结果由代码生成，未手工填写。

结构化决议：`docs/objective_risk_decision.json`；机器可读核算：`results/model_review/objective_risk_audit.json`。

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "scripts/audit_objective_risk.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.audit_objective_risk import compute_audit
result = compute_audit()
print(result["scope"])
print("SciPy", result["scipy_version"])

Pure mathematical diagnostics; no market scenario or archived result is read.
SciPy 1.18.1


## Results

### 1. λ 隐含的财富取舍与期限尺度

旧目标为 $J_H=\mathbb E\log(R_H/100)-\lambda\mathbb E D_H$。若平均 MDD 增加 0.01，维持同一目标值需要几何等价终值乘以 $\exp(0.01\lambda)$。

$$
\frac{J_H}{H}=\frac{\mathbb E\log(R_H/100)}{H}-\frac{\lambda}{H}\mathbb E D_H.
$$

固定 λ 表示固定的终值财富—回撤交换率；若希望固定年度交换率，有效年度系数却是 λ/H。两种偏好都可以定义，但不能混称。

In [2]:
tradeoffs = pd.DataFrame(result["lambda_tradeoffs"]).rename(columns={
    "lambda":"λ", "terminal_geometric_wealth_compensation":"终值补偿比例",
    "annual_factor_minus_one_H5":"5年：等效年度乘数减1",
    "annual_factor_minus_one_H15":"15年：等效年度乘数减1",
    "annual_factor_minus_one_H30":"30年：等效年度乘数减1"})
cols = ["λ", "终值补偿比例", "5年：等效年度乘数减1", "15年：等效年度乘数减1", "30年：等效年度乘数减1"]
display(tradeoffs[cols].style.hide(axis="index").format({c:"{:.4%}" for c in cols[1:]}))

λ,终值补偿比例,5年：等效年度乘数减1,15年：等效年度乘数减1,30年：等效年度乘数减1
10,10.5171%,2.0201%,0.6689%,0.3339%
4,4.0811%,0.8032%,0.2670%,0.1334%
1,1.0050%,0.2002%,0.0667%,0.0333%


上表为乘法补偿，**不是 CAGR 固定增加多少个百分点**。若原 CAGR 为 $g$，补偿后为 $(1+g)\exp(0.01\lambda/H)-1$。

这些偏好不能从市场数据识别。也不能比较不同 λ 下各自最大的 J 来反选 λ：因为 $D\ge0$，最优 J 天然随 λ 非增。λ 不是 CRRA 的 γ；取消该罚项后，log 目标仍是一项公开的规范性选择。

### 2. 平均回撤相同，尾部适配可以不同

所有结局的终值均回到 100：

- 稳定损失：必然经历 $100\to90\to100$。
- 罕见深跌：80% 为 $100\to100\to100$，20% 为 $100\to50\to100$。

两者的期望 log 终值、平均 MDD 相同，超限概率却不同；因此 λ 与 α 不是同一偏好的两种写法。

In [3]:
rows = [{"构造策略":label, "平均MDD":item["mean_mdd"], "P(MDD>25%)":item["exceedance_probability"],
         "期望log终值比":item["expected_log_terminal_ratio"]}
        for label, item in [("稳定损失", result["same_mean_mdd_counterexample"]["steady_loss"]),
                            ("罕见深跌", result["same_mean_mdd_counterexample"]["rare_deep_loss"])]]
display(pd.DataFrame(rows).style.hide(axis="index").format({"平均MDD":"{:.2%}", "P(MDD>25%)":"{:.2%}", "期望log终值比":"{:.3f}"}))

构造策略,平均MDD,P(MDD>25%),期望log终值比
稳定损失,10.00%,0.00%,0.000
罕见深跌,10.00%,20.00%,0.000


### 3. 相同超限概率可以掩盖不同严重程度

令 $L=20\%$。两种策略均有 90% 概率经历 10% 回撤；余下 10% 分别经历 21% 和 80% 回撤。所有路径最终回到 100。

$$
e_{H,L}=\mathbb E(D_H-L)_+=p_{H,L}\mathbb E[D_H-L\mid D_H>L].
$$

当超限概率非零时，平均超额回撤等于超限频率乘以条件平均超限幅度。计算它不代表已经约束它；真正的约束仍需要额外预算。

In [4]:
rows = [{"构造策略":label, "超限概率":item["exceedance_probability"],
         "平均超额回撤":item["expected_excess_mdd"], "CVaR90(MDD)":item["cvar_90_mdd"]}
        for label, item in [("小幅突破", result["same_exceedance_probability_counterexample"]["small_overshoot"]),
                            ("严重突破", result["same_exceedance_probability_counterexample"]["large_overshoot"])]]
display(pd.DataFrame(rows).style.hide(axis="index").format({c:"{:.2%}" for c in ["超限概率", "平均超额回撤", "CVaR90(MDD)"]}))

构造策略,超限概率,平均超额回撤,CVaR90(MDD)
小幅突破,10.00%,0.10%,21.00%
严重突破,10.00%,6.00%,80.00%


严重突破的平均超额回撤为 6%，小幅突破为 0.1%，相差 **60 倍**。主概率约束允许两者通过，不能据此声称控制全部严重尾部。

有限分布采用以下定义：

$$
\operatorname{CVaR}_{\beta}(D)=\min_{\eta}\left\{\eta+\frac{\mathbb E(D-\eta)_+}{1-\beta}\right\}.
$$

这里计算跨完整路径 MDD 分布的 CVaR，不是按年份混合的 CDaR。约束 $\operatorname{CVaR}_{.90}(D)\le L$ 可推出 $P(D>L)\le.10$，但额外要求最坏 10% 尾部的平均 MDD 不超过 L。它并不一定比 α=.05 的概率约束更强；匹配不同 α 时应写成 $\operatorname{CVaR}_{1-\alpha}(D)\le L$。

故将其保留为诊断和备选政策，不把问卷自动改写为更强承诺。变分定义能处理有限场景中的分位点原子；CVaR 的性质并不自动证明本题动态相对 MDD 优化为凸问题。[Rockafellar & Uryasev, 2002](https://sites.math.washington.edu/~rtr/papers/rtr187-CVaR2.pdf)

### 4. 10% 风险预算与 95% 统计证据不同

α 是容许的市场超限概率，δ 是统计区间构造的不覆盖频率。这里 δ=.05 仅作方法示例，同样不是问卷给定值。

固定规则在 n 条独立同分布评价路径上有 k 条超限，单侧 95% 精确上界为：

$$
U(k,n)=\operatorname{Beta}^{-1}(.95;k+1,n-k),\qquad k<n.
$$

零超限时 $U(0,n)=1-.05^{1/n}$。代码分别以 Beta 分位数和直接反演二项 CDF 核对。[NIST](https://www.itl.nist.gov/div898/software/dataplot/refman2/auxillar/exacbino.htm)

In [5]:
bounds = pd.DataFrame(result["fixed_rule_binomial_examples"])
bounds["超限/路径"] = bounds["exceedances"].astype(str) + "/" + bounds["independent_paths"].astype(str)
display(bounds[["超限/路径", "point_estimate", "one_sided_95_upper"]].rename(columns={
    "point_estimate":"样本超限率", "one_sided_95_upper":"单侧95%精确上界"}).style.hide(axis="index").format({"样本超限率":"{:.2%}", "单侧95%精确上界":"{:.3%}"}))
print("n=120且上界≤10%时最多超限数：", result["maximum_k_at_n120_for_upper_le_10pct"])
print("零超限时使上界≤10%的最小n：", result["minimum_zero_exceedance_n_for_upper_le_10pct"])

超限/路径,样本超限率,单侧95%精确上界
0/24,0.00%,11.735%
0/30,0.00%,9.503%
0/120,0.00%,2.466%
8/80,10.00%,17.320%
12/120,10.00%,15.698%
6/120,5.00%,9.630%
7/120,5.83%,10.677%


n=120且上界≤10%时最多超限数： 6
零超限时使上界≤10%的最小n： 29


这些数值不是对当前模型的认证。适用边界：

- 在同一批结果中挑出最有利的权重/规则后，普通逐点 CP 界不能自动认证被选方案；可用独立评价，或对事先固定的有限候选族建立同时界。
- 举例：预设 232 个固定策略，对同一 H,L 用 Bonferroni 修正，120 路径即便零超限，上界约为 6.79%，而不是逐点的 2.47%。这并未覆盖从数据学习预测库、动态规则和多客户声明的全部选择过程。
- 重叠训练折产生相关的测试误差。嵌套交叉验证可评价选择流程与稳定性，但不能把 OOF 超限数当成某个固定最终策略的独立二项试验；重复折数也不会增加独立场景数。[Bengio & Grandvalet, 2004](https://www.jmlr.org/papers/volume5/grandvalet04a/grandvalet04a.pdf)
- 生成更多预测路径能降低给定模型下的 Monte Carlo 误差，不能补出新的原始独立市场证据。

“证据不足”不等于“已经证明风险过高”；样本点估计通过也不等于“已经证明安全”。

### 5. 通胀与历史风险的数学边界

对同一期限、同一市场分布：

$$
\mathbb E\log(R_H^{\pi}/100)=\mathbb E\log(W_H^{\pi}/100)-\sum_{t=1}^H\mathbb E\log(1+i_t).
$$

比较策略时，共同通胀项抵消，不要求收益与通胀独立。实际 log 增长仍是合理购买力尺度，但不能说仅这个变换就改变了资产排序。实际财富 MDD 才在风险约束中反映购买力路径。

全程 $D_H=\max(D_{past},D_{continuation})$。过去已发生的 MDD 不可改变；之后低于旧最大值的两种损失可能得到相同全程 MDD。若过去已经超过 L，后续配置不能让全程从未超过 L 重新成立，不能重置历史后声称原约束恢复。

每年重新给 10% 的新超限额度也不等于全程 10%。若每年给定此前未超限的首次超限条件概率恒为 .10，则 30 年全程概率为 $1-.9^{30}\approx95.76\%$。本轮只确定完整期限约束，后续年度规则需评价完整滚动策略或明确传递条件风险预算。

In [6]:
cancellation = result["inflation_log_cancellation_example"]
display(pd.DataFrame([{
    "名义期望log差": cancellation["nominal_expected_log_gap"],
    "实际期望log差": cancellation["real_expected_log_gap"]
}]).style.hide(axis="index").format("{:.10f}"))
print("30年构造例的全程超限概率：", f'{result["repeated_annual_hazard_example"]["full_horizon_breach_probability"]:.4%}')
print("数学核验：", result["checks"])

名义期望log差,实际期望log差
0.0324137971,0.0324137971


30年构造例的全程超限概率： 95.7609%
数学核验： {'beta_quantile_reconciles_with_binomial_cdf_inversion': True, 'zero_exceedance_sample_size_boundary': True, 'counterexamples_recomputed_from_positive_wealth_paths': True, 'discrete_cvar_variational_definition_handles_atoms': True, 'partial_boundary_atom_cvar_27pct': True, 'equality_boundary_and_invalid_input_guards': True, 'inflation_term_cancels_without_independence_assumption': True}


## Takeaways

建议的总体模型：

$$
\max_{\pi\in\Pi}\quad G_H(\pi)=\frac1H\mathbb E\log(R_H^\pi/100)
\quad\text{s.t.}\quad P(D_H^\pi>L)\le\alpha.
$$

- Π 包含因果交易、费用及另行论证的流动性条件。入门适配仅在同一三产品菜单中比较，按各客户 H、L 评价，不额外设计第四款产品。
- 采用显式风险约束内追求 log 增长的思想，不照搬特殊 Kelly 模型的凸界。该文相对初始财富的损失与本题相对历史峰值的实际 MDD 不同。[Busseti, Ryu & Boyd, 2016](https://web.stanford.edu/~boyd/papers/pdf/kelly.pdf)
- α=.10 是公开服务基准，与 P90 汇报口径便于沟通。**题目要求报告 P90，不能推出客户接受 10% 超限概率。** 5%/10%/20% 情形须事先列明，不能按希望得到的映射选择。
- 同时报超限概率、平均超额回撤、CVaR 与增长分位数，展示增长—频率—严重度前沿。前沿能删除劣选项，但不能自动推出唯一 Balanced 代表点。
- 下轮实验前需统一定义样本满足且稳定、偏好敏感/证据不足、样本不满足及无合适产品时的转介。总体数学模型不等于已经验证的生产算法。
- 本轮不重新评估最终集、不改写旧报告。后续再用开发集 1–120 研究前沿与产品稳定性。

### 来源与核查范围

比赛原文 docs/problem_A.pdf，第 1–2 页给出问卷和评价要求，没有规定 λ、α 或统计置信水平。结构化决议保存各来源支持与不支持的范围。

脚本核验包括 Beta/二项反演一致、样本量边界、正财富路径反例、离散 CVaR 的部分分位点原子、等号不超限与非法输入处理、通胀项抵消。所有代码单元按顺序执行，输出来自数学计算。